In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier

data = pd.read_csv("../data/processed/model_data.csv", parse_dates=["cutoff"])

FEATURES = ["recency_days", "tenure_days", "frequency", "monetary", "avg_order_value",
            "purchases_last_90d", "revenue_last_90d",
            "purchases_last_180d", "revenue_last_180d", "avg_gap_days"]

train = data[data["cutoff"].isin(pd.to_datetime(["2010-09-01", "2010-12-01", "2011-03-01"]))]
gb = HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05,
                                    max_iter=200, random_state=42)
gb.fit(train[FEATURES], train["label"])

# placeholder assumptions (labelled as assumptions in the README)
N, RESP, COST, DISC = 500, 0.05, 2.0, 0.10
print(len(train), "training rows")        # expect 12050

12050 training rows


In [2]:
def r_score(d):   # same cut-offs as the SQL RFM view; 5 = most recent
    return np.select([d <= 20, d <= 59, d <= 186, d <= 409], [5, 4, 3, 2], default=1)

def f_score(f):
    return np.select([f == 1, f == 2, f <= 4, f <= 9], [1, 2, 3, 4], default=5)

def picks_for(df, p, n):
    y, aov = df["label"].values, df["avg_order_value"].values
    r, f = r_score(df["recency_days"].values), f_score(df["frequency"].values)
    elig = np.where((r <= 2) & (f >= 3))[0]        # "At risk" and "Cannot lose" at the cutoff
    return {
        "Highest past spend": np.argsort(-df["monetary"].values, kind="stable")[:n],
        "RFM at-risk rule": elig[np.argsort(-df["monetary"].values[elig], kind="stable")][:n],
        "Model: (1-p) x order value": np.argsort(-((1 - p) * aov), kind="stable")[:n],
        "Hindsight ceiling": np.argsort(-(aov * (1 - y)), kind="stable")[:n],
    }

def pool_stats(df, idx):
    y, aov = df["label"].values, df["avg_order_value"].values
    contrib = aov[idx] * (1 - y[idx])               # value we could win back from non-buyers
    pool = contrib.sum()
    top5 = np.sort(contrib)[::-1][:5].sum() / pool if pool > 0 else 0
    return {"contacted": len(idx), "non_buyer_rate": (1 - y[idx]).mean(),
            "value_pool": pool, "top5_share": top5}

def backtest(df, p, n):
    rows = []
    y, aov = df["label"].values, df["avg_order_value"].values
    # random contact: exact expectation, no simulation noise
    rows.append({"strategy": "Random", "contacted": n,
                 "non_buyer_rate": (1 - y).mean(),
                 "value_pool": n * (aov * (1 - y)).mean(), "top5_share": np.nan})
    for name, idx in picks_for(df, p, n).items():
        rows.append({"strategy": name, **pool_stats(df, idx)})
    out = pd.DataFrame(rows)
    return out

def add_money(out, resp=RESP, cost=COST, disc=DISC):
    out = out.copy()
    out["recovered"] = resp * (1 - disc) * out["value_pool"]
    out["cost"] = cost * out["contacted"]
    out["net"] = out["recovered"] - out["cost"]
    out["break_even_response"] = out["cost"] / ((1 - disc) * out["value_pool"])
    return out

In [3]:
results = []
for c in ["2011-06-01", "2011-09-01"]:
    df = data[data["cutoff"] == pd.Timestamp(c)].reset_index(drop=True)
    p = gb.predict_proba(df[FEATURES])[:, 1]
    out = add_money(backtest(df, p, N))
    out.insert(0, "cutoff", c)
    out.insert(1, "customers_known", len(df))
    results.append(out)
    print(c, "| known customers:", len(df), "| base rate:", round(df["label"].mean(), 3))
    print(out.drop(columns="cutoff").round(3).to_string(index=False), "\n")

summary = pd.concat(results, ignore_index=True)

2011-06-01 | known customers: 4908 | base rate: 0.329
 customers_known                   strategy  contacted  non_buyer_rate  value_pool  top5_share  recovered   cost       net  break_even_response
            4908                     Random        500           0.671  118321.139         NaN   5324.451 1000.0  4324.451                0.009
            4908         Highest past spend        500           0.226  183288.568       0.372   8247.986 1000.0  7247.986                0.006
            4908           RFM at-risk rule        500           0.786  152050.014       0.127   6842.251 1000.0  5842.251                0.007
            4908 Model: (1-p) x order value        500           0.756  427669.395       0.159  19245.123 1000.0 18245.123                0.003
            4908          Hindsight ceiling        500           1.000  503850.431       0.135  22673.269 1000.0 21673.269                0.002 

2011-09-01 | known customers: 5224 | base rate: 0.428
 customers_known          

In [4]:
import os
os.makedirs("../results", exist_ok=True)

rows = []
for c in ["2011-06-01", "2011-09-01"]:
    df = data[data["cutoff"] == pd.Timestamp(c)].reset_index(drop=True)
    p = gb.predict_proba(df[FEATURES])[:, 1]
    for n in (200, 500, 1000):
        base = backtest(df, p, n)
        for resp in (0.02, 0.05, 0.10):
            o = add_money(base, resp=resp)
            o.insert(0, "cutoff", c); o.insert(1, "n_target", n); o.insert(2, "response_rate", resp)
            rows.append(o[["cutoff", "n_target", "response_rate", "strategy", "contacted", "net"]])
sens = pd.concat(rows, ignore_index=True)

print(sens[sens["cutoff"] == "2011-09-01"]
      .pivot_table(index=["n_target", "response_rate"], columns="strategy", values="net")
      .round(0).to_string())

summary.to_csv("../results/backtest_summary.csv", index=False)
sens.to_csv("../results/backtest_sensitivity.csv", index=False)

strategy                Highest past spend  Hindsight ceiling  Model: (1-p) x order value  RFM at-risk rule   Random
n_target response_rate                                                                                              
200      0.02                       1416.0             5311.0                      4293.0            1365.0    318.0
         0.05                       4140.0            13878.0                     11333.0            4013.0   1394.0
         0.10                       8679.0            28156.0                     23065.0            8426.0   3188.0
500      0.02                       1767.0             7695.0                      5994.0            1861.0    794.0
         0.05                       5916.0            20737.0                     16484.0            6153.0   3485.0
         0.10                      12833.0            42474.0                     33968.0           13305.0   7969.0
1000     0.02                       2590.0            10121.0   

In [5]:
def controls(df, n):
    aov = df["avg_order_value"].values
    rfm_rank = ((-df["recency_days"]).rank(pct=True) + df["frequency"].rank(pct=True)
                + df["monetary"].rank(pct=True)).values / 3      # 0 to 1, higher = more likely to buy
    return {
        "Highest order value only": np.argsort(-aov, kind="stable")[:n],
        "RFM rank: (1-score) x order value": np.argsort(-((1 - rfm_rank) * aov), kind="stable")[:n],
    }

rows = []
for c in ["2011-06-01", "2011-09-01"]:
    df = data[data["cutoff"] == pd.Timestamp(c)].reset_index(drop=True)
    p = gb.predict_proba(df[FEATURES])[:, 1]
    allp = {**picks_for(df, p, N), **controls(df, N)}
    for name in ["Model: (1-p) x order value", "Highest order value only",
                 "RFM rank: (1-score) x order value", "Hindsight ceiling"]:
        idx = allp[name]
        s = pool_stats(df, idx)
        s.update(strategy=name, cutoff=c,
                 median_order_value=float(np.median(df["avg_order_value"].values[idx])))
        rows.append(s)

ctrl = add_money(pd.DataFrame(rows))
print(ctrl[["cutoff", "strategy", "contacted", "non_buyer_rate", "median_order_value",
            "value_pool", "net", "break_even_response"]].round(3).to_string(index=False))

    cutoff                          strategy  contacted  non_buyer_rate  median_order_value  value_pool       net  break_even_response
2011-06-01        Model: (1-p) x order value        500           0.756             755.908  427669.395 18245.123                0.003
2011-06-01          Highest order value only        500           0.558             884.091  381493.062 16167.188                0.003
2011-06-01 RFM rank: (1-score) x order value        500           0.814             678.700  422879.461 18029.576                0.003
2011-06-01                 Hindsight ceiling        500           1.000             672.817  503850.431 21673.269                0.002
2011-09-01        Model: (1-p) x order value        500           0.666             775.425  388533.933 16484.027                0.003
2011-09-01          Highest order value only        500           0.470             904.950  341116.491 14350.242                0.003
2011-09-01 RFM rank: (1-score) x order value        500

In [6]:
df = data[data["cutoff"] == pd.Timestamp("2011-09-01")].reset_index(drop=True)
p = gb.predict_proba(df[FEATURES])[:, 1]
base = backtest(df, p, N)

rows = []
for cost in (2, 10, 25):
    for resp in (0.02, 0.05, 0.10):
        o = add_money(base, resp=resp, cost=cost)
        rows.append(o.assign(cost_per_contact=cost, response_rate=resp)
                    [["cost_per_contact", "response_rate", "strategy", "net"]])

print(pd.concat(rows).pivot_table(index=["cost_per_contact", "response_rate"],
                                  columns="strategy", values="net").round(0).to_string())

strategy                        Highest past spend  Hindsight ceiling  Model: (1-p) x order value  RFM at-risk rule   Random
cost_per_contact response_rate                                                                                              
2                0.02                       1767.0             7695.0                      5994.0            1861.0    794.0
                 0.05                       5916.0            20737.0                     16484.0            6153.0   3485.0
                 0.10                      12833.0            42474.0                     33968.0           13305.0   7969.0
10               0.02                      -2233.0             3695.0                      1994.0           -2139.0  -3206.0
                 0.05                       1916.0            16737.0                     12484.0            2153.0   -515.0
                 0.10                       8833.0            38474.0                     29968.0            9305.0   3969.0


In [7]:
ctrl.to_csv("../results/backtest_controls.csv", index=False)
pd.concat(rows).to_csv("../results/backtest_cost_sensitivity.csv", index=False)

In [8]:
import json
import joblib

joblib.dump(gb, "../models/gb_repeat_purchase.joblib")
with open("../models/model_info.json", "w") as f:
    json.dump({"features": FEATURES,
               "target": "buys again within 90 days after the cutoff",
               "trained_on_cutoffs": ["2010-09-01", "2010-12-01", "2011-03-01"],
               "note": "probabilities were too low on the Sep 2011 test season; use for ranking"},
              f, indent=2)
print("saved")

saved
